In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedGroupKFold

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

In [2]:
df = pd.read_csv("../data/concrete_data.csv")
df = df.drop_duplicates().reset_index(drop=True)

print("Shape:", df.shape)

Shape: (1005, 9)


In [3]:
# Total cementitious material (everything that acts like cement)
df["total_binder"] = df["cement"] + df["slag"] + df["fly_ash"]

# Classic concrete ratios: more water per binder = weaker concrete
df["water_cement_ratio"] = df["water"] / df["cement"]
df["water_binder_ratio"] = df["water"] / df["total_binder"]

# Total aggregate (coarse + fine)
df["total_aggregate"] = df["coarse_aggregate"] + df["fine_aggregate"]

# Superplasticizer dose relative to binder
df["superplasticizer_binder_ratio"] = df["superplasticizer"] / df["total_binder"]

# Log of age: straightens the curing curve (fast early gain, slow later)
df["log_age"] = np.log(df["age"])

new_features = ["total_binder", "water_cement_ratio", "water_binder_ratio",
                "total_aggregate", "superplasticizer_binder_ratio", "log_age"]

df[new_features].describe().T.round(3)

,count,mean,std,min,25%,50%,75%,max
total_binder,1005.0,406.207,91.424,200.000,336.280,388.480,480.000,640.000
water_cement_ratio,1005.0,0.756,0.314,0.267,0.547,0.690,0.938,1.882
water_binder_ratio,1005.0,0.473,0.126,0.235,0.390,0.480,0.561,0.900
total_aggregate,1005.0,1747.063,102.230,1457.000,1679.000,1758.300,1829.000,1970.000
superplasticizer_binder_ratio,1005.0,0.014,0.013,0.000,0.000,0.015,0.024,0.057
log_age,1005.0,3.168,1.189,0.000,1.946,3.332,4.025,5.900


In [4]:
df[new_features + ["strength"]].corr()["strength"].drop("strength").sort_values().round(3)

water_binder_ratio              -0.611
water_cement_ratio              -0.489
total_aggregate                 -0.256
superplasticizer_binder_ratio    0.222
log_age                          0.563
total_binder                     0.598
Name: strength, dtype: float64

In [5]:
# Rows with the same 7 ingredient values = same mix (just tested at different ages)
ingredients = ["cement", "slag", "fly_ash", "water", "superplasticizer",
               "coarse_aggregate", "fine_aggregate"]

df["mix_id"] = df.groupby(ingredients).ngroup()

print("Total rows        :", len(df))
print("Unique mix designs:", df["mix_id"].nunique())

Total rows        : 1005
Unique mix designs: 428


In [6]:
# Age groups (rare ages like 1 and 120 days are combined with their neighbors)
df["age_group"] = pd.cut(df["age"], bins=[0, 7, 28, 100, 365],
                         labels=["1-7", "14-28", "56-100", "120-365"])

# Strength groups (4 equal-size groups)
df["strength_group"] = pd.qcut(df["strength"], q=4,
                               labels=["Low", "Medium", "High", "Very high"])

# Combine both into one label, e.g. "14-28_High"
df["strata"] = df["age_group"].astype(str) + "_" + df["strength_group"].astype(str)

# Merge very small strata (< 10 rows) into one "rare" group so every fold can get some
counts = df["strata"].value_counts()
rare = counts[counts < 10].index
df.loc[df["strata"].isin(rare), "strata"] = "rare"

df["strata"].value_counts()

strata
14-28_Medium         176
1-7_Low              164
14-28_High           124
56-100_Very high     117
14-28_Very high       96
14-28_Low             85
56-100_High           65
1-7_Medium            43
120-365_High          32
1-7_High              29
56-100_Medium         24
120-365_Very high     21
1-7_Very high         17
rare                  12
Name: count, dtype: int64

In [7]:
# 5 splits -> take the first one as the test set = 20% test
sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
train_idx, test_idx = next(sgkf.split(df, df["strata"], groups=df["mix_id"]))

df["split"] = "train"
df.loc[test_idx, "split"] = "test"

print(df["split"].value_counts())

# Check: no mix design should be in both train and test
common = set(df.loc[train_idx, "mix_id"]) & set(df.loc[test_idx, "mix_id"])
print("Mixes in both train and test:", len(common))

split
train    800
test     205
Name: count, dtype: int64
Mixes in both train and test: 0


In [8]:
print("Age group distribution (%):")
print((pd.crosstab(df["age_group"], df["split"], normalize="columns") * 100).round(1))
print()
print("Strength in train vs test:")
df.groupby("split")["strength"].describe().round(2)

Age group distribution (%):
split      test  train
age_group             
1-7        22.0   26.0
14-28      48.8   47.6
56-100     23.4   20.1
120-365     5.9    6.2

Strength in train vs test:


,count,mean,std,min,25%,50%,75%,max
split,,,,,,,,
test,205.0,36.01,15.99,6.28,24.89,34.20,46.24,81.75
train,800.0,35.06,16.36,2.33,23.12,33.74,44.65,82.60


In [9]:
train_df = df[df["split"] == "train"].reset_index(drop=True)
test_df = df[df["split"] == "test"].reset_index(drop=True)

cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)

# Each training row gets the name of the fold it is used to validate
train_df["fold"] = ""
for fold_number, (tr, val) in enumerate(cv.split(train_df, train_df["strata"],
                                                 groups=train_df["mix_id"]), start=1):
    train_df.loc[val, "fold"] = "fold_" + str(fold_number)

# Test rows are never used in cross-validation
test_df["fold"] = "test"

train_df["fold"].value_counts().sort_index()

fold
fold_1    154
fold_2    157
fold_3    167
fold_4    145
fold_5    177
Name: count, dtype: int64

In [10]:
print("Age group distribution per fold (%):")
print((pd.crosstab(train_df["age_group"], train_df["fold"], normalize="columns") * 100).round(1))
print()
print("Strength per fold:")
print(train_df.groupby("fold")["strength"].agg(["count", "mean", "std", "min", "max"]).round(2))
print()

# Check: no mix design should appear in a fold AND in the other folds
for f in sorted(train_df["fold"].unique()):
    val_mixes = set(train_df.loc[train_df["fold"] == f, "mix_id"])
    other_mixes = set(train_df.loc[train_df["fold"] != f, "mix_id"])
    print(f, "-> shared mixes with other folds:", len(val_mixes & other_mixes))

Age group distribution per fold (%):
fold       fold_1  fold_2  fold_3  fold_4  fold_5
age_group                                        
1-7          24.0    26.1    27.5    27.6    24.9
14-28        49.4    50.3    46.1    47.6    45.2
56-100       22.1    19.7    19.8    16.6    22.0
120-365       4.5     3.8     6.6     8.3     7.9

Strength per fold:
        count   mean    std   min    max
fold                                    
fold_1    154  36.86  17.11  4.83  82.60
fold_2    157  32.05  15.70  3.32  74.36
fold_3    167  36.92  17.56  2.33  80.20
fold_4    145  33.49  15.77  7.84  78.80
fold_5    177  35.68  15.22  6.81  74.19

fold_1 -> shared mixes with other folds: 0
fold_2 -> shared mixes with other folds: 0
fold_3 -> shared mixes with other folds: 0
fold_4 -> shared mixes with other folds: 0
fold_5 -> shared mixes with other folds: 0


In [11]:
final_df = pd.concat([train_df, test_df], ignore_index=True)

# Remove helper columns that were only needed for splitting
final_df = final_df.drop(columns=["age_group", "strength_group", "strata"])

output_path = "../data/concrete_data_folds.csv"
final_df.to_csv(output_path, index=False)

print("Saved to:", output_path)
print("Shape:", final_df.shape)
print()
print(final_df["fold"].value_counts().sort_index())

Saved to: ../data/concrete_data_folds.csv
Shape: (1005, 18)

fold
fold_1    154
fold_2    157
fold_3    167
fold_4    145
fold_5    177
test      205
Name: count, dtype: int64


In [12]:
final_df.head()

,cement,slag,fly_ash,water,superplasticizer,coarse_aggregate,fine_aggregate,age,strength,total_binder,water_cement_ratio,water_binder_ratio,total_aggregate,superplasticizer_binder_ratio,log_age,mix_id,split,fold
0,540.0,0.0,0.0,162.0,2.5,1040.0,676.0,28,79.986111,540.0,0.300000,0.30000,1716.0,0.00463,3.332205,425,train,fold_1
1,540.0,0.0,0.0,162.0,2.5,1055.0,676.0,28,61.887366,540.0,0.300000,0.30000,1731.0,0.00463,3.332205,426,train,fold_3
2,332.5,142.5,0.0,228.0,0.0,932.0,594.0,270,40.269535,475.0,0.685714,0.48000,1526.0,0.00000,5.598422,333,train,fold_1
3,332.5,142.5,0.0,228.0,0.0,932.0,594.0,365,41.052780,475.0,0.685714,0.48000,1526.0,0.00000,5.899897,333,train,fold_1
4,198.6,132.4,0.0,192.0,0.0,978.4,825.5,360,44.296075,331.0,0.966767,0.58006,1803.9,0.00000,5.886104,147,train,fold_4


In [13]:
df = pd.read_csv("../data/concrete_data_folds.csv")

print("Shape before:", df.shape)
print(df.columns.tolist())

weak_features = ["total_aggregate", "superplasticizer_binder_ratio"]

df = df.drop(columns=weak_features)

print("Removed:", weak_features)
print("Shape after:", df.shape)
print(df.columns.tolist())

output_path = "../data/concrete_data_folds.csv"

df.to_csv(output_path, index=False)
print("Saved to:", output_path)

# Verify
check = pd.read_csv(output_path)
print("Shape of saved file:", check.shape)
print()
print(check["fold"].value_counts().sort_index())

Shape before: (1005, 18)
['cement', 'slag', 'fly_ash', 'water', 'superplasticizer', 'coarse_aggregate', 'fine_aggregate', 'age', 'strength', 'total_binder', 'water_cement_ratio', 'water_binder_ratio', 'total_aggregate', 'superplasticizer_binder_ratio', 'log_age', 'mix_id', 'split', 'fold']
Removed: ['total_aggregate', 'superplasticizer_binder_ratio']
Shape after: (1005, 16)
['cement', 'slag', 'fly_ash', 'water', 'superplasticizer', 'coarse_aggregate', 'fine_aggregate', 'age', 'strength', 'total_binder', 'water_cement_ratio', 'water_binder_ratio', 'log_age', 'mix_id', 'split', 'fold']
Saved to: ../data/concrete_data_folds.csv
Shape of saved file: (1005, 16)

fold
fold_1    154
fold_2    157
fold_3    167
fold_4    145
fold_5    177
test      205
Name: count, dtype: int64
